[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/semantica-agi/semantica/blob/main/cookbook/introduction/24_Change_Management.ipynb)

# Change Management — Practical Guide

Semantica's `change_management` module provides versioning, audit trails, and data-integrity checks for knowledge graphs and ontologies:

- **`ChangeLogEntry`** — standardized change metadata (validated timestamp/author)
- **`InMemoryVersionStorage` / `SQLiteVersionStorage`** — version snapshot storage with named tags
- **`compute_checksum` / `verify_checksum`** — SHA-256 integrity verification

This notebook runs a complete save → tag → verify → tamper-detect cycle. All outputs are real executed results verified against the repository's `semantica/change_management/` source at the time of writing (the `pip install` cell may fetch a newer release with slightly different behavior).

In [1]:
!pip install -q semantica

## 1) A `ChangeLogEntry` records *who* changed *what*, *when*

`author` must be a valid email — the dataclass validates on construction (`ValidationError` otherwise), which keeps audit trails clean.

In [2]:
from semantica.change_management import ChangeLogEntry

entry = ChangeLogEntry(
    timestamp="2026-08-15T09:00:00Z",
    author="demo@example.com",
    description="initial version",
)
entry

ChangeLogEntry(timestamp='2026-08-15T09:00:00Z', author='demo@example.com', description='initial version', change_id=None, related_changes=[])

## 2) Save a versioned snapshot

A snapshot is a dict with a required `label` plus your payload. Here we attach the KG data, the change log, and a SHA-256 `checksum` computed over everything except the checksum field itself.

In [3]:
from semantica.change_management import InMemoryVersionStorage, compute_checksum

storage = InMemoryVersionStorage()

snapshot = {
    "label": "v1.0.0",
    "data": {"entities": {"acme": {"type": "Company"}}},
    "change_log": {
        "timestamp": entry.timestamp,
        "author": entry.author,
        "description": entry.description,
    },
}
snapshot["checksum"] = compute_checksum({k: v for k, v in snapshot.items() if k != "checksum"})

storage.save(snapshot)
storage.exists("v1.0.0")

True

## 3) Named tags pin a version for releases

`save_tag` / `get_tag` map stable names (e.g. `release`) to version labels, decoupling consumers from label churn.

In [4]:
storage.save_tag("release", "v1.0.0")

storage.get_tag("release"), [s["label"] for s in storage.list_all()]

('v1.0.0', ['v1.0.0'])

## 4) Verify integrity — and catch tampering

`verify_checksum(snapshot)` recomputes the SHA-256 over the snapshot (minus its `checksum` field) and compares. A single mutated character in the data flips the result to `False`.

In [5]:
from semantica.change_management import verify_checksum

stored = storage.get("v1.0.0")
print("intact:", verify_checksum(stored))

tampered = storage.get("v1.0.0")
tampered["data"]["entities"]["acme"]["note"] = "mutated after the fact"
print("tampered:", verify_checksum(tampered))

intact: True
tampered: False


## 5) Retiring a version

`delete(label)` removes a snapshot; tags pointing at it are your responsibility to update.

In [6]:
storage.delete("v1.0.0")
storage.exists("v1.0.0")

False

## Summary

| Task | API |
|---|---|
| Record audit metadata | `ChangeLogEntry(timestamp, author=email, description)` |
| Persist a version | `InMemoryVersionStorage().save({"label": ..., ...})` |
| Pin a release name | `save_tag("release", "v1.0.0")` / `get_tag("release")` |
| Integrity check | `compute_checksum(snap)` / `verify_checksum(snap)` |
| Persistent backend | `SQLiteVersionStorage(path)` — same interface |

See also `semantica/change_management/change_management_usage.md` for the manager classes (`TemporalVersionManager`, `OntologyVersionManager`).